# NB_dv_metadata — DV 2.0 Shared Helpers

Provides shared utilities used by all vault notebooks:
- `load_model()`: reads the dv_model.json config
- `generate_hash_key()` / `generate_diff_hash()`: PySpark hash expressions
- `create_hub_table()`, `create_link_table()`, `create_sat_table()`: DDL helpers
- `create_pit_table()`, `create_bridge_table()`: business vault DDL
- `get_latest_diff_hash()`: window function for satellite CDC

In [ ]:
dbutils.widgets.text("MODEL_PATH", "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA", "vault", "Vault schema name")

MODEL_PATH   = dbutils.widgets.get("MODEL_PATH")
CATALOG      = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA = dbutils.widgets.get("VAULT_SCHEMA")

In [ ]:
import json
from pathlib import Path

VOLUME_ROOT = Path("/Volumes/workspace/default/mnt")

def load_model(model_path: str) -> dict:
    """Load dv_model.json — supports absolute paths or relative (resolved under VOLUME_ROOT)."""
    p = Path(model_path)
    if not p.is_absolute():
        p = VOLUME_ROOT / model_path
    return json.loads(p.read_text())

model = load_model(MODEL_PATH)
print(f"Loaded model: {len(model['hubs'])} hubs, {len(model['links'])} links, {len(model['satellites'])} satellites")


In [ ]:
from pyspark.sql import functions as F

def generate_hash_key(bk_cols: list) -> F.Column:
    """Generate SHA-256 hash key from business key columns."""
    return F.sha2(
        F.concat_ws("||", *[F.upper(F.trim(F.col(c).cast("string"))) for c in bk_cols]),
        256,
    )

def generate_diff_hash(tracked_cols: list) -> F.Column:
    """Generate SHA-256 diff hash from tracked satellite columns."""
    return F.sha2(
        F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in tracked_cols]),
        256,
    )

In [ ]:
def create_hub_table(hub_cfg: dict) -> None:
    """Create a hub Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{hub_cfg['name'].lower()}"
    bk_cols = ', '.join(f"{c} STRING" for c in hub_cfg['business_key_columns'])
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            HK_{hub_cfg['name'][4:]} STRING NOT NULL,
            {bk_cols},
            LOAD_DATE TIMESTAMP,
            RECORD_SOURCE STRING
        ) USING DELTA
    """)

def create_link_table(lnk_cfg: dict) -> None:
    """Create a link Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{lnk_cfg['name'].lower()}"
    fk_cols = ', '.join(f"HK_{r['hub'][4:]} STRING" for r in lnk_cfg['hub_references'])
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            LHK_{lnk_cfg['name'][4:]} STRING NOT NULL,
            {fk_cols},
            LOAD_DATE TIMESTAMP,
            RECORD_SOURCE STRING
        ) USING DELTA
    """)

def create_sat_table(sat_cfg: dict) -> None:
    """Create a satellite Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{sat_cfg['name'].lower()}"
    tracked_cols = ', '.join(f"{c} STRING" for c in sat_cfg['tracked_columns'])
    parent_hub = sat_cfg['parent_hub']
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            HK_{parent_hub[4:]} STRING NOT NULL,
            LOAD_DATE TIMESTAMP NOT NULL,
            DIFF_HK STRING,
            {tracked_cols},
            RECORD_SOURCE STRING
        ) USING DELTA
    """)

In [ ]:
def create_pit_table(pit_cfg: dict) -> None:
    """Create a PIT Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{pit_cfg['name'].lower()}"
    hub_name = pit_cfg['hub']
    sat_cols = '\n'.join(f"    {s}_LDTS TIMESTAMP,\n    {s}_HK STRING," for s in pit_cfg['satellites'])
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            HK_{hub_name[4:]} STRING NOT NULL,
            SNAPSHOT_DATE DATE NOT NULL,
            {sat_cols}
            LOAD_DATE TIMESTAMP
        ) USING DELTA
    """)

def create_bridge_table(brg_cfg: dict) -> None:
    """Create a bridge Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{brg_cfg['name'].lower()}"
    hk_cols = '\n'.join(f"    HK_{p[4:]} STRING," for p in brg_cfg['path'] if p.startswith('HUB_'))
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            {hk_cols}
            LOAD_DATE TIMESTAMP
        ) USING DELTA
    """)

In [ ]:
from pyspark.sql import Window

def get_latest_diff_hash(sat_table: str, hk_col: str) -> 'DataFrame':
    """Return the latest DIFF_HK per hub key from an existing satellite table.

    Uses a window function to find the most recent record per hub key,
    which is then LEFT JOINed against incoming data to detect changes.
    """
    w = Window.partitionBy(hk_col).orderBy(F.col('LOAD_DATE').desc())
    return (
        spark.table(sat_table)
        .withColumn('_rn', F.row_number().over(w))
        .filter(F.col('_rn') == 1)
        .select(hk_col, 'DIFF_HK')
    )